# 03 — Product Embeddings

Generate and save CLIP image and Multilingual E5-small text embeddings for product search and recommendations.

## 1 · Dependencies

In [5]:
%pip -q install "transformers>=4.44,<5" pandas pyarrow pillow requests tqdm

## 2 · Imports, Drive and configuration

In [7]:
import os, re, json, pickle, time, hashlib, gc
from io import BytesIO
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, AutoModel
from google.colab import drive

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ElectroLens")
OUT = ROOT / "search_assets"
PARTS = OUT / "image_parts"
OUT.mkdir(parents=True, exist_ok=True)
PARTS.mkdir(parents=True, exist_ok=True)

CLIP_ID = "openai/clip-vit-base-patch32"
E5_ID = "intfloat/multilingual-e5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_BATCH = 32
E5_BATCH = 64
IMAGE_WORKERS = 8
TOP_K = 10
print("Device:", DEVICE)
if DEVICE == "cuda": print(torch.cuda.get_device_name(0))
else: print("CPU works but embedding generation will be slower. T4 GPU recommended.")

Mounted at /content/drive
Device: cuda
Tesla T4


## 3 · Load and validate the **final** product catalog

In [8]:
products = pd.read_parquet(ROOT / "products_final.parquet").reset_index(drop=True)
required = {"parent_asin", "title", "image_url"}
missing = required - set(products.columns)
if missing: raise ValueError(f"Missing columns: {missing}. Inspect products.columns.")
if products.parent_asin.isna().any() or not products.parent_asin.is_unique:
    raise ValueError("parent_asin must be unique and non-null.")
products["parent_asin"] = products.parent_asin.astype(str)
products["title"] = products.title.fillna("").astype(str)
asins = products.parent_asin.to_numpy(dtype=str)
catalog_hash = hashlib.sha256("\n".join(asins).encode()).hexdigest()[:16]
meta_path = OUT / "manifest.json"
manifest = {"catalog_hash": catalog_hash, "n_products": len(products),
            "clip_model": CLIP_ID, "e5_model": E5_ID}
if meta_path.exists():
    old = json.loads(meta_path.read_text())
    if old != manifest:
        raise ValueError("Catalog/model changed. Use a new OUT directory; do not mix old checkpoints.")
else:
    meta_path.write_text(json.dumps(manifest, indent=2))
products.to_parquet(OUT / "products_search.parquet", index=False)
print("Products:", len(products), "| catalog fingerprint:", catalog_hash)
print("Columns:", products.columns.tolist())
display(products[["parent_asin", "title", "image_url"]].head(3))

Products: 13328 | catalog fingerprint: 4f1e28ecb05ad3eb
Columns: ['parent_asin', 'title', 'main_group', 'subcategory', 'cat_path', 'store', 'price', 'average_rating', 'rating_number', 'features', 'description', 'image_url', 'product_group', 'kw_ok']


,parent_asin,title,image_url
0,B08XPWDSWW,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,https://m.media-amazon.com/images/I/41fBo0aKiI...
1,B08XNCHTCY,TOZO T6 True Wireless Earbuds Bluetooth 5.3 He...,https://m.media-amazon.com/images/I/31XCpv4iHk...
2,B07S764D9V,"Panasonic ErgoFit Wired Earbuds, In-Ear Headph...",https://m.media-amazon.com/images/I/41mcJ3vWMN...


## 4 · Product text

E5 uses `passage: ...` for indexed products and `query: ...` for user searches. This prefix is **important** for E5.

Use title + category + a short description when available. The description is truncated to avoid expensive processing and noisy product text.

In [9]:
def clean_field(x, limit=350):
    if x is None or isinstance(x, (list, dict)): return ""
    s = re.sub(r"\s+", " ", str(x)).strip()
    return "" if s.lower() in {"nan", "none", "null"} else s[:limit]

def make_product_text(row):
    fields = [clean_field(row.get("title"), 180)]
    for name in ("store", "product_group", "description"):
        if name in products.columns:
            value = clean_field(row.get(name), 250)
            if value: fields.append(value)
    return " | ".join(fields)

product_texts = products.apply(make_product_text, axis=1).tolist()
print(product_texts[0])

TOZO T10 Bluetooth 5.3 Wireless Earbuds with Wireless Charging Case IPX8 Waterproof Stereo Headphones in Ear Built in Mic Headset Premium Sound with Deep Bass for Sport White (2022 | TOZO | headphones


## 5 · Load CLIP and E5 encoders (once)

In [10]:
clip = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
e5_tokenizer = AutoTokenizer.from_pretrained(E5_ID)
e5 = AutoModel.from_pretrained(E5_ID).to(DEVICE).eval()

CLIP_DIM = int(clip.config.projection_dim)
E5_DIM = int(e5.config.hidden_size)
print("CLIP dim:", CLIP_DIM, "| E5 dim:", E5_DIM)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

CLIP dim: 512 | E5 dim: 384


## 6 · Embedding functions (normalized float32)

In [11]:
@torch.inference_mode()
def encode_clip_images(images):
    x = clip_processor(images=images, return_tensors="pt").to(DEVICE)
    z = clip.get_image_features(**x)
    return F.normalize(z.float(), p=2, dim=1).cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_clip_text(texts):
    x = clip_processor(text=texts, padding=True, truncation=True, max_length=77,
                       return_tensors="pt").to(DEVICE)
    z = clip.get_text_features(**x)
    return F.normalize(z.float(), p=2, dim=1).cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_e5(texts, prefix="passage: "):
    batch = e5_tokenizer([prefix + t for t in texts], padding=True, truncation=True,
                         max_length=256, return_tensors="pt").to(DEVICE)
    out = e5(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1)
    z = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(z.float(), p=2, dim=1).cpu().numpy().astype("float32")

## 7 · Create E5 product embeddings (cached on Drive)

In [12]:
e5_file = OUT / "e5_embeddings.npy"
if e5_file.exists():
    e5_vectors = np.load(e5_file)
    if e5_vectors.shape != (len(products), E5_DIM):
        raise ValueError("Cached E5 embeddings have the wrong shape.")
else:
    chunks = []
    for start in tqdm(range(0, len(products), E5_BATCH), desc="E5 product embeddings"):
        chunks.append(encode_e5(product_texts[start:start+E5_BATCH]))
    e5_vectors = np.ascontiguousarray(np.vstack(chunks))
    np.save(e5_file, e5_vectors)
print("E5 vectors:", e5_vectors.shape)

E5 product embeddings:   0%|          | 0/209 [00:00<?, ?it/s]

E5 vectors: (13328, 384)


## 8 · Download images temporarily

We do not store image files on Drive. Failed URLs are recorded as unavailable. Each completed batch is stored as a small checkpoint so an interrupted Colab run can resume.

In [13]:
def fetch_image(url):
    if not isinstance(url, str) or not url.startswith(("https://", "http://")):
        return None
    try:
        r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        if len(r.content) > 12_000_000: return None
        with Image.open(BytesIO(r.content)) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
            img.thumbnail((1024, 1024))
            return img.copy()
    except (requests.RequestException, OSError, UnidentifiedImageError, ValueError):
        return None

def fetch_batch(urls):
    with ThreadPoolExecutor(max_workers=IMAGE_WORKERS) as pool:
        return list(pool.map(fetch_image, urls))

## 9 · CLIP image embeddings with resumable batch files

In [14]:
image_vectors = np.zeros((len(products), CLIP_DIM), dtype="float32")
image_valid = np.zeros(len(products), dtype=bool)

for start in tqdm(range(0, len(products), CLIP_BATCH), desc="CLIP product images"):
    end = min(start + CLIP_BATCH, len(products))
    part = PARTS / f"batch_{start:06d}_{end:06d}.npz"
    if part.exists():
        with np.load(part) as d:
            rows, vecs = d["rows"], d["vectors"]
    else:
        images = fetch_batch(products.image_url.iloc[start:end].tolist())
        good = [i for i, img in enumerate(images) if img is not None]
        rows = np.asarray([start + i for i in good], dtype="int64")
        vecs = encode_clip_images([images[i] for i in good]) if good else np.empty((0, CLIP_DIM), dtype="float32")
        # Atomic file replacement: a failed write won't leave a valid-looking checkpoint.
        tmp = str(part) + ".tmp.npz"
        np.savez_compressed(tmp, rows=rows, vectors=vecs)
        os.replace(tmp, part)
    if len(rows):
        image_vectors[rows] = vecs
        image_valid[rows] = True

np.save(OUT / "clip_image_embeddings.npy", image_vectors)
np.save(OUT / "clip_image_valid.npy", image_valid)
print("Valid image embeddings:", int(image_valid.sum()), "/", len(products))
print("Missing/failed images:", int((~image_valid).sum()))

CLIP product images:   0%|          | 0/417 [00:00<?, ?it/s]

Valid image embeddings: 13328 / 13328
Missing/failed images: 0


## 10 · Verify saved embedding assets

`image_parts/` checkpoints support resuming failed image downloads.

In [15]:
for name in ["products_search.parquet", "e5_embeddings.npy", "clip_image_embeddings.npy", "clip_image_valid.npy", "manifest.json"]:
    path = OUT / name
    print(f"{name}: {path.stat().st_size / 1024**2:.2f} MB" if path.exists() else f"MISSING: {name}")
print("Saved to:", OUT)


products_search.parquet: 12.55 MB
e5_embeddings.npy: 19.52 MB
clip_image_embeddings.npy: 26.03 MB
clip_image_valid.npy: 0.01 MB
manifest.json: 0.00 MB
Saved to: /content/drive/MyDrive/ElectroLens/search_assets


In [16]:
print("Products:", len(products))
print("E5 embeddings:", e5_vectors.shape)
print("CLIP embeddings:", image_vectors.shape)
print("Valid images:", image_valid.sum())
print("Failed images:", (~image_valid).sum())

Products: 13328
E5 embeddings: (13328, 384)
CLIP embeddings: (13328, 512)
Valid images: 13328
Failed images: 0
